In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from imutils import paths
import cv2

In [2]:
os.chdir(r"C:\Users\heman\OneDrive\Desktop\AFTER BTECH\innomatics\ml\Plant_D\rice leaf diseases dataset\rice leaf diseases dataset")

In [3]:
os.listdir()

['Bacterialblight', 'Brownspot', 'Leafsmut']

In [4]:

l=[]
for i in os.listdir():
    img_num_data=np.empty((0,270000))
    k=0
    for j in os.listdir(i):
        img=cv2.imread(r"C:\Users\heman\OneDrive\Desktop\AFTER BTECH\innomatics\ml\Plant_D\rice leaf diseases dataset\rice leaf diseases dataset"+"/"+i+"/"+j)
        img=cv2.resize(src=img,dsize=(300,300),interpolation=cv2.INTER_LINEAR)
        img=np.array(img)
        flatten_img=img.flatten().reshape(1,-1)
        img_num_data=np.vstack((img_num_data,flatten_img))
    temp_df=pd.DataFrame(img_num_data)
    temp_df['Target']=i
    l.append(temp_df)
df=pd.concat(l)


KeyboardInterrupt: 

In [7]:
df

,0,1,2,3,4,5,6,7,8,9,...,269991,269992,269993,269994,269995,269996,269997,269998,269999,Target
0,21.0,125.0,168.0,20.0,125.0,168.0,21.0,127.0,168.0,20.0,...,10.0,109.0,69.0,7.0,109.0,67.0,7.0,109.0,67.0,Bacterialblight
1,8.0,108.0,66.0,9.0,109.0,67.0,11.0,108.0,68.0,12.0,...,21.0,127.0,168.0,21.0,126.0,167.0,20.0,125.0,166.0,Bacterialblight
2,17.0,74.0,41.0,19.0,74.0,41.0,21.0,73.0,43.0,25.0,...,88.0,215.0,176.0,89.0,216.0,177.0,72.0,201.0,162.0,Bacterialblight
3,70.0,202.0,161.0,85.0,217.0,176.0,89.0,219.0,178.0,91.0,...,17.0,72.0,39.0,16.0,71.0,38.0,15.0,72.0,39.0,Bacterialblight
4,15.0,56.0,29.0,13.0,57.0,28.0,1.0,51.0,21.0,23.0,...,167.0,248.0,223.0,174.0,255.0,230.0,167.0,251.0,223.0,Bacterialblight
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,20.0,22.0,22.0,20.0,22.0,22.0,20.0,22.0,22.0,20.0,...,44.0,61.0,52.0,44.0,61.0,50.0,44.0,61.0,50.0,Leafsmut
1456,96.0,165.0,122.0,96.0,165.0,122.0,96.0,165.0,122.0,95.0,...,92.0,79.0,81.0,92.0,79.0,81.0,92.0,79.0,81.0,Leafsmut
1457,20.0,95.0,44.0,24.0,98.0,50.0,23.0,98.0,52.0,20.0,...,28.0,87.0,83.0,24.0,83.0,79.0,22.0,82.0,76.0,Leafsmut
1458,5.0,38.0,31.0,6.0,39.0,32.0,8.0,39.0,30.0,9.0,...,1.0,38.0,22.0,0.0,37.0,19.0,0.0,32.0,13.0,Leafsmut


In [41]:
df.to_csv(r"C:\Users\heman\OneDrive\Desktop\AFTER BTECH\innomatics\ml\Plant_D\Flattened_Images.csv")

In [2]:
df=pd.read_csv(r"C:\Users\heman\OneDrive\Desktop\AFTER BTECH\innomatics\ml\Plant_D\Flattened_Images.csv")

MemoryError: 

In [ ]:
df

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [9]:
X_train,X_test,y_train,y_test=train_test_split(df.drop('Target',axis=1),df['Target'],test_size=0.2,random_state=42)

In [10]:
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((3747, 270000), (3747,), (937, 270000), (937,))

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Image-optimized grid (smaller, focused ranges)
param_grid = {
    'max_depth': [5, 8, 10, 12],
    'min_samples_split': [20, 50, 100],
    'min_samples_leaf': [10, 25, 50],
    'criterion': ['gini']  # entropy slower on images
}

dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')  # for imbalanced image classes
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)  # HOG/flattened pixels/TF-IDF visual words

print("Best:", grid_search.best_params_)
# Typical result: {'max_depth': 8, 'min_samples_leaf': 25, 'min_samples_split': 50}


MemoryError: Unable to allocate 6.03 GiB for an array with shape (2998, 270000) and data type float64

In [13]:
import pickle
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,f1_score
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
f1 = f1_score(y_test, dt.predict(X_test), average='weighted')
print("F1 Score:", f1)


F1 Score: 0.929725808142521


In [14]:
accuracy_score(y_test, dt.predict(X_test))

0.9295624332977588

In [18]:
import pickle as pkl
with open ("rice_leaf_diseases.pkl",'wb') as f:
    pkl.dump(dt,f)

In [19]:
df["Target"].value_counts()

Target
Brownspot          1620
Bacterialblight    1604
Leafsmut           1460
Name: count, dtype: int64

In [20]:
with open('rice_leaf_diseases.pkl', 'rb') as f:
    model = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'rice_leaf_diseases.pkl'